This is a classic question that tests:
 Understanding of window functions.
 Ability to detect consecutive patterns without timestamp.
 Logic building for sequence-based data problems.

This type of problem is frequently asked in SQL interviews for Data Engineer roles to check your pattern-recognition skills

In [0]:
dt=[("A"),("A"),("B"),("C"),("A"),("A"),("A"),("C"),("D"),("D"),("D"),("B"),("B"),("C")]
df=spark.createDataFrame(dt)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
df=df.withColumn("element",F.col("_1"))
df.createOrReplaceTempView("df_v")
df = df.withColumn("id", F.monotonically_increasing_id()).drop(F.col("_1"))

df=df.withColumn("rank_id",F.row_number().over(Window.orderBy(F.col("id"))))\
    .withColumn("denserank",F.dense_rank().over(Window.orderBy(F.col("element"))))\
        .withColumn("grp_id",F.col("rank_id")-F.col("denserank"))
w1 = Window.orderBy("id")
w2 = Window.orderBy("element")
df2 = (df
    .withColumn("rn", F.row_number().over(w1))
    .withColumn("dr", F.dense_rank().over(w2))
    .withColumn("grp_id", F.col("rn") - F.col("dr"))
)

# 4. Group by element + grp_id and find runs of 3+
result = (df2
    .groupBy("element", "grp_id")
    .count()
    .filter("count >= 3")
    .select("element")
    .distinct()
)

result.show()


In [0]:
#spark.sql("""select * from df_v""").show()
spark.sql("""with cte1 as (select *,row_number() over (order by 1) rank_id
           from df_v)
           ,cte2 as (
           select * ,dense_rank() over (partition by element order by rank_id) dense_rn
           from cte1),
           cte3 as (
           select *,rank_id-dense_rn grp_id from cte2),
           cte4 as (
           select distinct element,count(grp_id) from cte3 group by element,grp_id having count(grp_id)>2)
           select * from cte3 order by rank_id """).show()
spark.sql("""with cte1 as (select *,row_number() over (order by 1) rank_id
           from df_v)
           ,cte2 as (
           select * ,dense_rank() over (partition by element order by rank_id) dense_rn
           from cte1),
           cte3 as (
           select *,rank_id-dense_rn grp_id from cte2)
           select distinct element,count(grp_id) from cte3 group by element,grp_id having count(grp_id)>2  """).show()